# Lab 2: Q-Learning from Scratch 🧠
### Master's Degree Programme in Applications of AI to Industry and Communications
#### Subject: Reinforcement Learning and Generative AI
##### University of Alcalá

Welcome to your second lab assignment in the Reinforcement Learning module! In this notebook, **you will build your first Reinforcement Learning agent entirely from scratch**. 

Instead of relying on a high-level library like Stable-Baselines3, you will implement the foundational **Q-Learning** algorithm using pure Python and NumPy to solve the FrozenLake-v1 ❄️ and Taxi-v3 🚕 environments.

> [!TIP]
> **💡 Originality & Research Bonus:**
> Original solutions and deeper research are highly valued and rewarded in this lab! Students who investigate alternative exploration strategies (e.g., Softmax/Boltzmann exploration, Upper Confidence Bound exploration), experiment with dynamic learning rate decay schedules, test custom reward structures, or perform systematic hyperparameter comparisons will receive bonus marks. Show your initiative by exploring beyond the baseline!

Throughout this notebook, you will encounter interleaved theoretical questions. Please answer them in the designated markdown cells.

Let's get started!


## 1. Setting up the Computational Environment 🖥️
We will begin by installing the necessary dependencies.
- `gymnasium`: Provides the environments.
- `pygame`: Used for environment UI rendering.
- `numpy`: Used for matrix operations and our Q-table memory.


In [ ]:
!pip install -q gymnasium pygame numpy huggingface_hub imageio imageio_ffmpeg pyglet tqdm

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3-opengl ffmpeg xvfb
!pip install -q pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 2. Importing Packages 📦
In addition to the environment and math libraries, we import `random` for our stochastic policies and `imageio` to record replay videos.


In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import pickle
from tqdm.notebook import tqdm

# Set global random seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 3. Theoretical Foundations: Q-Learning 📚

Before writing code, let's review the mathematics of Q-Learning.

Q-Learning is a **Value-Based**, **Off-Policy** algorithm. Instead of learning the policy directly, it learns a *Q-Function* $Q(s, a)$. The Q-Function estimates the expected cumulative reward of taking action $a$ in state $s$ and acting optimally thereafter.

Since our state space and action space are both discrete and small, we can represent this Q-Function as a literal table (the **Q-Table**) with rows corresponding to states and columns corresponding to actions.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

### The Bellman Update Equation
As the agent explores the environment, it observes transitions: $(S_t, A_t, R_{t+1}, S_{t+1})$. It then updates its Q-Table using the Bellman Equation:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha \left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$$

Where:
- $\alpha$ (Learning Rate): How much we accept the new information versus the old.
- $\gamma$ (Discount Factor): How much we care about future rewards versus immediate rewards.

---
### 📝 Question 1: The Bellman Equation Components
In the update equation above, there is a term inside the brackets: $\left[ R_{t+1} + \gamma \max_{a} Q(S_{t+1}, a) - Q(S_t, A_t) \right]$.
**This term is known as the TD Error (Temporal Difference Error). Explain in your own words what the TD Error represents conceptually. Why do we subtract the current $Q(S_t, A_t)$ from the target?**

*Your answer here...*


## 4. Part 1: Frozen Lake ⛄

Let's instantiate our first environment: [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/). The agent must navigate from Start (S) to Goal (G) by walking on frozen tiles (F) and avoiding holes (H).

We will start with the deterministic version: `is_slippery=False`.


In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # Get a random observation

The Observation Space is `Discrete(16)`. The state is represented as a single integer (0 to 15) indicating the grid position.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">


In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

The Action Space is `Discrete(4)`: LEFT (0), DOWN (1), RIGHT (2), UP (3).

The Reward Function is highly sparse:
- Reach goal: +1
- Reach hole: 0
- Reach frozen: 0

### Initializing the Q-Table
We need a table of size `(state_space, action_space)`. Let's initialize it with zeros.


In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [ ]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

---
### 📝 Question 2: Q-Table Initialization
**Why is it common practice to initialize the Q-Table with zeros? Imagine we initialized the entire table with a large positive number (e.g., +100) instead. How would this alter the agent's exploratory behavior during the first few episodes?**

*Your answer here...*


## 5. Defining the Policies 🤖

Q-Learning is *off-policy*. We use an **$\epsilon$-greedy policy** to explore during training, but we learn the values of the strictly **greedy policy** (the $\max_a$ in the Bellman equation).

### The Greedy Policy (Exploitation)
This policy always selects the action with the highest Q-value for the given state.


In [ ]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action =

  return action

### The Epsilon-Greedy Policy (Exploration vs. Exploitation)
With probability $1 - \epsilon$, we exploit (use the greedy policy). With probability $\epsilon$, we explore (choose a completely random action).

Over time, we will decay $\epsilon$ so the agent explores less and exploits more.


In [ ]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num =
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action =
  # else --> exploration
  else:
    action = # Take a random action

  return action

---
### 📝 Question 3: Epsilon Decay
**Explain the necessity of decaying $\epsilon$ over time rather than keeping it at a constant value (e.g., $\epsilon = 0.5$). What would happen to the agent's performance in the final stages of training if $\epsilon$ never decreased?**

*Your answer here...*


## 6. Training the Q-Learning Agent 🏃

Here are the hyperparameters for our FrozenLake training run.


In [ ]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7          # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"     # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate
eval_seed = []               # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05            # Minimum exploration probability
decay_rate = 0.0005            # Exponential decay rate for exploration prob

Now, implement the training loop. You will need to utilize your `epsilon_greedy_policy` to select actions, and implement the Bellman update equation manually inside the loop.


In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action =

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info =

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] =

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

Let's look at the resulting Q-table. You should see non-zero values indicating the expected returns of the paths discovered by the agent.


In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

## 7. Evaluating the Agent 📈
We will now evaluate the agent deterministically (using the greedy policy only).


In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

> [!IMPORTANT]
> **Stochastic Environments:**
> The non-slippery FrozenLake environment is deterministic, making it easy to achieve a mean reward of 1.0. In the real world, actions have uncertain outcomes. When `is_slippery=True`, the agent might slip and move perpendicular to its intended direction.

---
### 📝 Question 4: Handling Stochasticity
**In a stochastic environment (like the slippery FrozenLake), the agent does not always end up in the state it intended. How does the Q-learning algorithm implicitly handle this stochasticity without explicitly knowing or modeling the transition probabilities of the environment?**

*Your answer here...*


### 🎥 Visualizing Learning Progression in FrozenLake-v1: Beginning, Middle & Final

Let's observe the evolution of the Q-Learning agent on the slippery 4x4 frozen lake across three distinct training milestones:
1. **Beginning (0 episodes / Untrained):** Random exploration; slips and falls into the very first ice hole 🕳️. The failure frame is frozen for 2.5 seconds so you can examine the exact moment of failure.
2. **Middle (~1,500 episodes / Partially Trained):** Navigates through the first rows, but misjudges a turn and slips into an obstacle hole near the bottom. Frozen for 2.5 seconds.
3. **Final (10,000 episodes / Converged):** Successfully executes the safe policy, avoids every hole, and reaches the goal Frisbee 🎁!


In [ ]:
import imageio
import base64
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import HTML

def add_fl_banner(frame, text, bg_color=(200, 30, 30), text_color=(255, 255, 255)):
    img = Image.fromarray(frame).convert("RGBA")
    w, h = img.size
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    banner_h = 32
    draw.rectangle([(0, h - banner_h), (w, h)], fill=(*bg_color, 215))
    draw.text((12, h - banner_h + 8), text, fill=text_color)
    combined = Image.alpha_composite(img, overlay)
    return np.array(combined.convert("RGB"))

def record_frozenlake_progression_video(Q_matrix, filename, is_slippery=False, seed=42, max_steps=100, fps=5, hold_seconds=0.8):
    env_eval = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=is_slippery, render_mode="rgb_array")
    state, info = env_eval.reset(seed=seed)
    frames = [env_eval.render()]
    total_r = 0
    step_count = 0
    terminated = truncated = False

    for step in range(max_steps):
        step_count = step + 1
        if Q_matrix is None or np.all(Q_matrix[state] == 0):
            action = env_eval.action_space.sample()
        else:
            action = int(np.argmax(Q_matrix[state]))
        state, reward, terminated, truncated, info = env_eval.step(action)
        total_r += reward
        frames.append(env_eval.render())
        if terminated or truncated:
            break
    env_eval.close()

    is_goal = (total_r > 0)
    status_text = f"🎁 GOAL REACHED! (Step {step_count})" if is_goal else f"🕳️ FELL IN HOLE (Failed at step {step_count})"
    banner_bg = (34, 139, 34) if is_goal else (178, 34, 34)

    # Freeze on the failure / goal frame so the viewer can examine the outcome
    if frames:
        terminal_frame = add_fl_banner(frames[-1], status_text, bg_color=banner_bg)
        frames[-1] = terminal_frame
        hold_frames = int(fps * hold_seconds)
        for _ in range(hold_frames):
            frames.append(terminal_frame)

    imageio.mimsave(filename, frames, fps=fps)
    print(f"🎬 Saved {filename} | {status_text} | Duration: {len(frames)/fps:.1f}s")
    return total_r, step_count

# 1. Beginning Phase: Untrained Q-table (0 episodes)
print("--- 1. Recording Untrained FrozenLake (0 episodes) ---")
rew_fl_untrained, _ = record_frozenlake_progression_video(None, "frozenlake_untrained.mp4", is_slippery=False, seed=42)

# 2. Middle Phase: Partially trained Q-table (~1,500 episodes)
print("\n--- 2. Training Mid-Checkpoint FrozenLake (1,500 episodes) ---")
Qtable_fl_mid = train(1500, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake.copy(), learning_rate, gamma)
rew_fl_mid, _ = record_frozenlake_progression_video(Qtable_fl_mid, "frozenlake_mid.mp4", is_slippery=False, seed=42)

# 3. Final Phase: Fully trained Q-table (from Section 4)
print("\n--- 3. Recording Final Converged FrozenLake (10,000 episodes) ---")
rew_fl_final, _ = record_frozenlake_progression_video(Qtable_frozenlake, "frozenlake_final.mp4", is_slippery=False, seed=42)

b64_fl_untrained = base64.b64encode(open("frozenlake_untrained.mp4", "rb").read()).decode("utf-8")
b64_fl_mid = base64.b64encode(open("frozenlake_mid.mp4", "rb").read()).decode("utf-8")
b64_fl_final = base64.b64encode(open("frozenlake_final.mp4", "rb").read()).decode("utf-8")

HTML(f"""
<div style="font-family: Arial, sans-serif; text-align: center; margin: 15px 0;">
  <h3 style="color: #1a1a1a;">❄️ FrozenLake-v1 Learning Progression Across Training Phases</h3>
  <div style="display: flex; justify-content: space-around; flex-wrap: wrap; gap: 15px; margin-top: 15px;">
    
    <!-- 1. Untrained -->
    <div style="flex: 1; min-width: 260px; background: #fff1f0; border: 1px solid #ffa39e; border-radius: 8px; padding: 12px;">
      <h4 style="color: #cf1322; margin-top: 0;">1. Beginning (0 ep) 💥</h4>
      <video width="100%" height="220" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_fl_untrained}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Untrained:</b> Slips into ice hole in first steps. Pauses on hole frame for 2.5s.</p>
    </div>

    <!-- 2. Mid -->
    <div style="flex: 1; min-width: 260px; background: #fffbe6; border: 1px solid #ffe58f; border-radius: 8px; padding: 12px;">
      <h4 style="color: #d48806; margin-top: 0;">2. Middle (~1.5k ep) 🟡</h4>
      <video width="100%" height="220" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_fl_mid}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Partially Trained:</b> Bypasses first rows, but slips near bottom. Pauses 2.5s on hole.</p>
    </div>

    <!-- 3. Final -->
    <div style="flex: 1; min-width: 260px; background: #f6ffed; border: 1px solid #b7eb8f; border-radius: 8px; padding: 12px;">
      <h4 style="color: #389e0d; margin-top: 0;">3. Final (10k ep) 🎁🏆</h4>
      <video width="100%" height="220" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_fl_final}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Fully Trained:</b> Solved policy; bypasses all holes and reaches Frisbee goal.</p>
    </div>

  </div>
</div>
""")


## 8. Part 2: Taxi-v3 - Independent Challenge 🚖

Now it is time to apply what you've learned to a much more complex environment. In [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/), the agent must navigate to a passenger, pick them up, and drop them off at a designated location.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">


In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

The state space is large (500 discrete states), encoding the taxi's position, the passenger's position, and the destination.


In [ ]:
# Create our Q table with state_size rows and action_size columns (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Q-table shape: ", Qtable_taxi .shape)

### Hyperparameter Tuning Challenge
For this environment, **you must determine the optimal hyperparameters yourself**. Think carefully about the number of states, how much exploration is needed, and the discount factor.

> [!TIP]
> - Since the state space is larger, you will likely need **more training episodes** than FrozenLake.
> - Since the episode can be long, $\gamma$ should be chosen carefully to propagate rewards backwards.
> - Make sure the decay rate allows the agent to explore long enough before $\epsilon$ becomes negligible.


In [ ]:
# Training parameters
# TODO: Define the number of training episodes and the learning rate
n_training_episodes =  # <YOUR_VALUE>
learning_rate =  # <YOUR_VALUE>

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# DO NOT MODIFY EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148]

# Environment parameters
env_id = "Taxi-v3"           # Name of the environment
max_steps = 99               # Max steps per episode
# TODO: Define the discount rate (gamma)
gamma =  # <YOUR_VALUE>

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
# TODO: Define minimum epsilon and decay rate
min_epsilon =  # <YOUR_VALUE>
decay_rate =  # <YOUR_VALUE>


In [ ]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

## 9. Assignment Delivery 🚀
You must publish your trained Taxi-v3 model to the Hugging Face Hub. To pass the assignment, you must achieve a `mean_reward >= 4.5` on the leaderboard evaluation.


In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json

In [ ]:
def record_video(env, Qtable, out_directory, fps=1):
  """
  Generate a replay video of the agent
  :param env
  :param Qtable: Qtable of our agent
  :param out_directory
  :param fps: how many frame per seconds (with taxi, we want to slow down)
  """
  images = []  
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0,500))
  img = env.render()
  images.append(img)
  while not terminated and not truncated:
    # Take the action (index) that have the maximum expected future reward given that state
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action) # We directly put next_state = state for recording logic
    img = env.render()
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def push_to_hub(
    repo_id, model, env, video_fps=1, local_repo_path="hub"
):
    """
    Evaluate, Generate a video and Upload a model to Hugging Face Hub.
    This method does the complete pipeline:
    - It evaluates the model
    - It generates the model card
    - It generates a replay video of the agent
    - It pushes everything to the Hub

    :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
    :param env
    :param video_fps: how many frame per seconds to record our video replay
    (with taxi-v3 and frozenlake-v1 we use 1)
    :param local_repo_path: where the local repository is
    """
    _, repo_name = repo_id.split("/")

    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
    )

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    # Pickle the model
    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and build JSON with evaluation metrics
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )

    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat()
    }

    # Write a JSON file called "results.json" that will contain the
    # evaluation results
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")

    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{env_id}**
  This is a trained model of a **Q-Learning** agent playing **{env_id}** .

  ## Usage

  ```python

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  ```
  """

    evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

    readme_path = repo_local_path / "README.md"
    readme = ""
    print(readme_path.exists())
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7. Push everything to the Hub
    api.upload_folder(
        repo_id=repo_id,
        folder_path=repo_local_path,
        path_in_repo=".",
    )

    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

### Step 1️⃣ — Create a Hugging Face Account and Generate a Write Token

> 🔑 **What is a Hugging Face token?**
> Hugging Face is a platform for sharing AI models and datasets. A **write token** authenticates your account and allows the notebook to publish your trained Taxi-v3 Q-table and evaluation video to the Hub automatically.

**To get your token:**
1. Go to [https://huggingface.co](https://huggingface.co) and create a free account (or log in if you already have one).
2. Once logged in, click on your profile picture (top-right) → **Settings** → **Access Tokens** in the left sidebar.
   - Direct link: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Click **"New token"**, choose a name (e.g., `rl-lab-token`), set the **Role** to **`Write`**, and click **"Generate a token"**.
4. **Copy the token** — it starts with `hf_...`. Store it securely (you will not be able to retrieve it again after leaving the page).

> ⚠️ **Important:** Keep your token private! Never share it or commit it to a public repository. It grants write access to your Hugging Face account.

Run the cell below to authenticate:


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

2️⃣ Push the Taxi-v3 model to the Hub.


In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

In [ ]:
username = "" # FILL THIS
repo_name = "" # FILL THIS
push_to_hub(
    repo_id=f"{username}/{repo_name}",
    model=model,
    env=env)

## 10. Visualizing the Learning Progression: Beginning, Middle & Final 🎥

Let's observe the evolution of the Q-Table in the **Taxi-v3** environment across three training milestones:
1. **Beginning (Untrained, 0 episodes):** Empty Q-table (all zeros); the taxi takes random decisions, makes illegal pickup/drop-off attempts (-10 penalty), and hits the 100-step timeout limit. The final timeout frame is held for 2.5 seconds.
2. **Middle (~4,000 episodes / Partially Trained):** Taxi learns passenger pickup, but wanders taking inefficient detours or attempts drop-off early. Held for 2.5s.
3. **Final (25,000 episodes / Converged):** Fully trained optimal policy; executes the shortest direct path to pickup and drop-off (+20 reward).


In [ ]:
import imageio
import base64
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import HTML

def add_taxi_banner(frame, text, bg_color=(200, 30, 30), text_color=(255, 255, 255)):
    img = Image.fromarray(frame).convert("RGBA")
    w, h = img.size
    overlay = Image.new("RGBA", img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    banner_h = 32
    draw.rectangle([(0, h - banner_h), (w, h)], fill=(*bg_color, 215))
    draw.text((12, h - banner_h + 8), text, fill=text_color)
    combined = Image.alpha_composite(img, overlay)
    return np.array(combined.convert("RGB"))

def record_taxi_progression_video(Q_matrix, filename="taxi_replay.mp4", max_ep_steps=100, seed=42, fps=6, hold_seconds=0.8):
    env_eval = gym.make("Taxi-v3", render_mode="rgb_array")
    state, info = env_eval.reset(seed=seed)
    frames = [env_eval.render()]
    total_r = 0
    step_count = 0
    terminated = truncated = False

    for step in range(max_ep_steps):
        step_count = step + 1
        if Q_matrix is None or np.all(Q_matrix[state] == 0):
            action = env_eval.action_space.sample()
        else:
            action = int(np.argmax(Q_matrix[state]))
        state, reward, terminated, truncated, info = env_eval.step(action)
        total_r += reward
        frames.append(env_eval.render())
        if terminated or truncated:
            break
    env_eval.close()

    is_success = (reward == 20)
    if is_success:
        status_text = f"🚕 PASSENGER DELIVERED! (+20 reward | {step_count} st)"
        banner_bg = (34, 139, 34)
    elif truncated or step_count >= max_ep_steps:
        status_text = f"⏳ TIME OUT (Score: {total_r:.0f} | 100 steps limit)"
        banner_bg = (178, 34, 34)
    else:
        status_text = f"💥 WRONG DROP-OFF (Score: {total_r:.0f} | Step {step_count})"
        banner_bg = (204, 102, 0)

    # Freeze on final failure / delivery frame for 2.5s so student can examine the outcome
    if frames:
        terminal_frame = add_taxi_banner(frames[-1], status_text, bg_color=banner_bg)
        frames[-1] = terminal_frame
        hold_frames = int(fps * hold_seconds)
        for _ in range(hold_frames):
            frames.append(terminal_frame)

    imageio.mimsave(filename, frames, fps=fps)
    print(f"🎬 Saved: {filename} | Total Reward: {total_r:.1f} | {status_text} | Duration: {len(frames)/fps:.1f}s")
    return total_r, step_count

# 1. Beginning Phase: Untrained Agent (all zeros Q-table)
print("--- 1. Recording Untrained Taxi (0 episodes) ---")
rew_taxi_untrained, _ = record_taxi_progression_video(None, "taxi_untrained.mp4", seed=SEED)

# 2. Middle Phase: Partially Trained Agent (~4,000 episodes)
print("\n--- 2. Training Mid-Checkpoint Taxi (~4,000 episodes) ---"
)
Qtable_taxi_mid = train(4000, min_epsilon_taxi, max_epsilon_taxi, decay_rate_taxi, env, max_steps_taxi, np.zeros_like(Qtable_taxi), learning_rate_taxi, gamma_taxi)
rew_taxi_mid, _ = record_taxi_progression_video(Qtable_taxi_mid, "taxi_mid.mp4", seed=SEED)

# 3. Final Phase: Fully Trained Agent (25,000 episodes)
print("\n--- 3. Recording Fully Trained Taxi (25,000 episodes) ---"
)
rew_taxi_final, _ = record_taxi_progression_video(Qtable_taxi, "taxi_final.mp4", seed=SEED)

b64_taxi_untrained = base64.b64encode(open("taxi_untrained.mp4", "rb").read()).decode("utf-8")
b64_taxi_mid = base64.b64encode(open("taxi_mid.mp4", "rb").read()).decode("utf-8")
b64_taxi_final = base64.b64encode(open("taxi_final.mp4", "rb").read()).decode("utf-8")

HTML(f"""
<div style="font-family: Arial, sans-serif; text-align: center; margin: 15px 0;">
  <h3 style="color: #1a1a1a;">🚕 Taxi-v3 Learning Progression Across Training Phases</h3>
  <div style="display: flex; justify-content: space-around; flex-wrap: wrap; gap: 15px; margin-top: 15px;">
    
    <!-- Untrained -->
    <div style="flex: 1; min-width: 260px; background: #fff1f0; border: 1px solid #ffa39e; border-radius: 8px; padding: 12px;">
      <h4 style="color: #cf1322; margin-top: 0;">1. Beginning (0 ep) 💥</h4>
      <video width="100%" height="240" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_taxi_untrained}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Untrained:</b> Random navigation, illegal actions, timeout at 100 steps. Pauses on timeout frame for 2.5s.</p>
    </div>

    <!-- Middle -->
    <div style="flex: 1; min-width: 260px; background: #fffbe6; border: 1px solid #ffe58f; border-radius: 8px; padding: 12px;">
      <h4 style="color: #d48806; margin-top: 0;">2. Middle (~4k ep) 🟡</h4>
      <video width="100%" height="240" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_taxi_mid}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Partially Trained:</b> Learns passenger pickup, but wanders taking inefficient detours. Brief 0.8s hold.</p>
    </div>

    <!-- Final -->
    <div style="flex: 1; min-width: 260px; background: #f6ffed; border: 1px solid #b7eb8f; border-radius: 8px; padding: 12px;">
      <h4 style="color: #389e0d; margin-top: 0;">3. Final (25k ep) 🚕🏆</h4>
      <video width="100%" height="240" controls autoplay loop muted style="border-radius: 6px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
        <source src="data:video/mp4;base64,{b64_taxi_final}" type="video/mp4">
      </video>
      <p style="font-size: 13px; color: #555; margin-top: 8px;"><b>Fully Trained:</b> Direct optimal path to pickup and delivery (+20 reward).</p>
    </div>

  </div>
</div>
""")


### 🎯 Tournament Scoring: Point Formula (Taxi)

The score obtained by your agent is the cumulative reward per episode:

$$R_{\text{Taxi}} = 20 \cdot \mathbf{1}_{\{\text{delivered}\}} - 1 \cdot (\text{steps}) - 10 \cdot (\text{illegal\_actions})$$

- **$+20$ pts:** Successfully delivering the passenger at the destination.
- **$-1$ pt / step:** Cost per step taken (rewarding shortest direct route).
- **$-10$ pts:** Penalty for each illegal pickup or drop-off attempt.



In [ ]:
import pickle

# Save our model dictionary to a file
with open("q-learning.pkl", "wb") as f:
    pickle.dump(model, f)


In [ ]:
%%writefile agent.py
import os
import numpy as np
import pickle

# Load the model
model_path = os.path.join(os.path.dirname(__file__), "q-learning.pkl")
with open(model_path, "rb") as f:
    model = pickle.load(f)
    
Qtable = model["qtable"]

def agent(observation):
    # Select the action with the maximum expected reward
    action = np.argmax(Qtable[observation][:])
    return int(action)


In [ ]:
!zip submission.zip q-learning.pkl agent.py

### 🚀 Option A: Submit to the Class Tournament via Pull Request (Recommended)

You can automatically fork the repository, upload your agent, and open a **Pull Request** to the class tournament leaderboard with a single click!

#### What happens when you run the cell below?

1. **🔀 Fork:** The script automatically creates a personal copy (*fork*) of `gramuah/RL_tournament` under your GitHub account using your GitHub token.
2. **📦 Upload:** Your `agent.py` and `q_table.pkl` files are uploaded to `submissions/lab2/<your_github_username>/` inside your fork.
3. **🔃 Pull Request:** A Pull Request (PR) is opened from your fork to the main repository.
4. **🤖 Automatic CI Evaluation:** A GitHub Actions robot immediately runs your agent for **30 episodes** on fixed, fair seeds and posts a report card directly as a comment on your PR (within ~2 minutes). The report includes mean reward, success rate, and performance tier.
5. **📊 Leaderboard Update:** Once an instructor reviews and merges your PR, your score appears on the [official tournament website](https://gramuah.github.io/RL_tournament/) and leaderboard.

> **Note:** If you submit again later (e.g., after improving your Q-table or hyperparameters), just re-run this cell. The script will update your existing fork branch and PR automatically.

#### Prerequisite: GitHub Personal Access Token (PAT)

> 🔑 **What is a GitHub PAT?**
> It is a password substitute that the script uses to create the fork and upload files on your behalf, without ever seeing your actual GitHub password.

**To generate one:**
1. Go to [https://github.com/settings/tokens](https://github.com/settings/tokens) (log in to GitHub if needed).
2. Click **"Generate new token (classic)"**.
3. Give it a **name** (e.g., `rl-tournament-lab`), set **Expiration** to at least 30 days, and check the **`repo`** checkbox (full control of private repositories).
4. Click **"Generate token"** at the bottom of the page.
5. **Copy the token** immediately — it starts with `ghp_...`. You will not be able to see it again.
6. Paste it in the secure prompt below (it will not be displayed on screen).


In [ ]:
import getpass
import os
import urllib.request

# 1. Load the automated submission tool from the repository
_submit_candidates = [
    os.path.join("..", "..", "tournament", "submit.py"),
    os.path.join("tournament", "submit.py"),
]
_loaded = False
for _candidate in _submit_candidates:
    if os.path.exists(_candidate):
        with open(_candidate, "r", encoding="utf-8") as _f:
            exec(_f.read())
        _loaded = True
        break
if not _loaded:
    try:
        _code = urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/main/tournament/submit.py", timeout=30).read().decode("utf-8")
        if "_MAINTAINERS" not in _code:
            _code = urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/chore/rename-lab-notebooks/tournament/submit.py", timeout=30).read().decode("utf-8")
        exec(_code)
    except Exception:
        exec(urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/main/tournament/submit.py", timeout=30).read().decode("utf-8"))

# 2. Enter your GitHub Token securely
GITHUB_TOKEN = getpass.getpass("🔑 Enter your GitHub Personal Access Token: ")

# 3. Automatically fork, upload, and open the Pull Request!
pr_url = submit_lab(
    lab_id=2,
    files_to_submit={
        "agent.py": "agent.py",
        "q-learning.pkl": "q-learning.pkl"
    },
    github_token=GITHUB_TOKEN
)

### 🥊 Head-to-Head Agent Duel (Compare your agent against another student or a baseline)

Want to see **who completes the trip faster in Taxi-v4**, or how your agent handles exactly the same challenge as a classmate's agent?

You can compare your model and view the visualizations in the following ways:

1. **🌐 In the Interactive GitHub Pages Web Arena (One Click, No Code):**
   Visit the official public portal: **[https://gramuah.github.io/RL_tournament/](https://gramuah.github.io/RL_tournament/)**.
   Use the dropdown menus to select any classmate or baseline and watch the **synchronized split-screen video replay**, real-time telemetry, and speed rankings.

2. **⚙️ Run Duels on Demand in GitHub Actions:**
   In **[Actions > Run Agent Duel](https://github.com/gramuah/RL_tournament/actions/workflows/duel.yml)**, click *"Run workflow"* to have the GitHub bot simulate the duel and publish an animated GIF in the run summary.

3. **🎬 Automatic Animations in [LEADERBOARD.md](https://github.com/gramuah/RL_tournament/blob/main/LEADERBOARD.md):**
   The class leaderboard includes embedded animated GIFs that play directly on the GitHub page.

4. **💻 Directly in This Notebook (Code Below):**
   Run the cell below to load the comparison tool, match your `agent.py` against any opponent (a baseline or a classmate's GitHub username), and watch the synchronized video right here in Colab.

In [ ]:
# ==============================================================================
# 🥊 HEAD-TO-HEAD AGENT DUEL (Taxi-v3 / Taxi-v4)
# ==============================================================================
import os
import urllib.request

# 1. Download and import the official comparison tool
_compare_candidates = [
    os.path.join("..", "..", "tournament", "compare.py"),
    os.path.join("tournament", "compare.py"),
]
_loaded = False
for _candidate in _compare_candidates:
    if os.path.exists(_candidate):
        with open(_candidate, "r", encoding="utf-8") as _f:
            exec(_f.read())
        _loaded = True
        break
if not _loaded:
    exec(urllib.request.urlopen("https://raw.githubusercontent.com/gramuah/RL_tournament/main/tournament/compare.py").read().decode("utf-8"))

# 2. Choose your opponent:
# - Available baseline: "baseline_random"
# - Or enter the GitHub username of any classmate who has uploaded their model!
OPPONENT = "baseline_random"

# 3. Run the duel (both use the same seed and initial configuration)
duel_stats = duel_agents(
    lab_id=2,
    agent_a="agent.py",           # Your trained agent
    agent_b=OPPONENT,             # Your opponent
    episodes=5,                   # Number of head-to-head races
    seed=42,
    output_video="duel_p2.mp4"
)

# 4. Play the synchronized video right here in Colab
display_duel("duel_p2.mp4")

### 📁 Option B: Deliver your Notebook to Virtual Campus (Blackboard)

Submitting to the GitHub tournament (Option A) is **recommended** but **not mandatory**. Even if you skip the Pull Request, your assignment is valid as long as it is submitted on Blackboard.

To submit on Blackboard:
1. Make sure **all cells have been executed** and their outputs are visible. Do **not** clear outputs before submitting — the instructor needs to see the training results and evaluation metrics.
   - ⚠️ **Important note about outputs in Google Colab:** Colab does **not** save cell outputs by default when you close the session. Before downloading, go to **Edit → Notebook settings** (or **Runtime → Run all**) to ensure all cells are re-executed and their outputs are visible. Then go to **File → Download → Download .ipynb** to save the notebook **with outputs**.
2. Rename the file following the convention: `Surname_Name_Lab2.ipynb` (e.g. `Garcia_Juan_Lab2.ipynb`).
3. Upload the `.ipynb` file through the **Blackboard** assignment box corresponding to Lab 2, before the deadline indicated by the instructor.

> ✅ **If you submitted to GitHub (Option A):** You do not need to re-submit the model files separately on Blackboard. Just upload the completed notebook file.
>
> ⚠️ **If you only submit on Blackboard (without GitHub):** This is accepted, but your agent will not appear on the class tournament leaderboard. We encourage you to submit to both!


________________________________________________________________________
**Congratulations on completing your Q-Learning Master's assignment!** 

You have implemented a foundational RL algorithm entirely from scratch, handled the exploration-exploitation trade-off, tuned hyperparameters independently for a complex environment, and tackled theoretical questions regarding Bellman equations and stochasticity. 

Ensure your inline questions are answered and your model is uploaded to the Hub before finalizing your submission.
